In [1]:
import os, time, logfire
from dotenv import load_dotenv
load_dotenv()


print(f"Gemini_Key: {' ✅'if os.getenv("Gemini_API") else 'Missing ❌'}")

print(f"Logfire Key: {'✅'if os.getenv("Logfire_API") else 'Missing ❌'}")

print(f"Open_AI Key: {'✅'if os.getenv("Open_AI_API") else 'Missing ❌'}")
print(f"Groq Key: {'✅'if os.getenv("Groq_API") else 'Missing ❌'}")
print(f"Tavily_API Key: {'✅'if os.getenv("Tavily_API") else 'Missing ❌'}")

SyntaxError: f-string: expecting a valid expression after '{' (2014261.py, line 6)

In [9]:
from pydantic import BaseModel
from typing import Optional

# Mock data



In [10]:
class LLmRequests(BaseModel):
    user_id: str
    session_id: str
    query: str
    model: str
    temperature: float = 0.7
    max_tokens: Optional[int] = None

class LLMResponse(BaseModel):
    answer: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    model_used: str

#pass data



In [11]:
request=LLmRequests(
    user_id="syed",
    session_id="session_test123",
    query="What is FIne Tuning..",
    model="llama-3.3-70b-versatile",
    max_tokens=540


)

In [12]:
response=LLMResponse(answer="Fine Tuning is a method of training part of your base model by updating weights in forward and backward propagation.",
                     input_tokens=19,
                     output_tokens=140,
                     latency_ms=333.2,
                     model_used="llama-3.3-70b-versatile")

In [15]:
logfire.configure()

Logfire project URL: https://logfire-us.pydantic.dev/syednoormujassum/first-logfire

In [16]:
with logfire.span("LLM_Call_Tracing",
                  userid=request.user_id,
                  session_id = request.session_id,
                  query=request.query,
                  LLM_model=request.model):
    logfire.info("Request is received", **request.model_dump())

    time.sleep(5)

    logfire.info("Response Sent", **response.model_dump())


    print(response)





22:52:26.498 LLM_Call_Tracing
22:52:26.499   Request is received
22:52:31.505   Response Sent
answer='Fine Tuning is a method of training part of your base model by updating weights in forward and backward propagation.' input_tokens=19 output_tokens=140 latency_ms=333.2 model_used='llama-3.3-70b-versatile'


## AUTO INSTRUMENTATION

Instrumenrtation is a way where logfire writes the spans automatically

ChatOpenAI of Langchain supports both groq and google gemini base url hence we can use chatopenai wrapper.

ChatGroq is hardcoded to Groq rest api url hence we cannot use other than Groq models. Same to google

In [42]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

logfire.instrument_openai()

llm_groq = ChatOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("Groq_API"),
    #model="qwen/qwen3.8-27b",
    model="meta-llama/llama-prompt-guard-2-86m",
    temperature=0.3
)

In [24]:
print(f"Calling Groq : {llm_groq.model}")

# below will automatically log everything in logfire
response=llm_groq.invoke([HumanMessage(content="Expalin what is Fine Tuning in 2 sentences only")])

print (response.content)

Calling Groq : qwen/qwen3.8-27b
23:11:17.955 Chat Completion with 'qwen/qwen3.8-27b' [LLM]
Fine-tuning is the process of further training a pre-trained machine learning model on a specific dataset to adapt it to a particular task or domain. This technique allows the model to leverage its general knowledge while specializing to achieve higher performance on targeted applications.


In [20]:
from groq import Groq

client = Groq(api_key=os.environ["Groq_API"])
[m.id for m in client.models.list().data]

['whisper-large-v3',
 'allam-2-7b',
 'canopylabs/orpheus-v1-english',
 'whisper-large-v3-turbo',
 'meta-llama/llama-prompt-guard-2-86m',
 'openai/gpt-oss-safeguard-20b',
 'meta-llama/llama-prompt-guard-2-22m',
 'qwen/qwen3.8-27b',
 'openai/gpt-oss-120b',
 'canopylabs/orpheus-arabic-saudi',
 'openai/gpt-oss-20b']

In [25]:
llm_groq.invoke([HumanMessage(content="what is span in oberservability")])

23:14:20.636 Chat Completion with 'qwen/qwen3.8-27b' [LLM]


AIMessage(content='In the context of observability, specifically within **distributed tracing**, a **Span** is the fundamental unit of work.\n\n### Simple Definition\nA **Span** represents a single, discrete operation or task within a request’s lifecycle. It has a start time, an end time, and metadata (tags/attributes) that describe what happened during that operation.\n\nThink of it this way:\n- A **Trace** is the entire journey of a user request (e.g., "User clicks \'Buy Now\'").\n- A **Span** is each individual step in that journey (e.g., "Validate cart," "Check inventory," "Charge credit card," "Update database").\n\n---\n\n### Key Characteristics of a Span\n1. **Start and End Time**: Every span records when it began and when it finished, allowing you to calculate its **duration**.\n2. **Name**: A human-readable label describing the operation (e.g., `GET /api/products`, `DB.query`, `Redis.get`).\n3. **Parent-Child Relationship**: Spans are often nested. A span can have a **parent s

Similarly lets try with Google gemini flash or Open AI gpt

In [28]:
gemini_llm=ChatOpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.getenv("Gemini_API"),
    temperature=0.5,
    model="gemini-3.5-flash-lite"

)

In [29]:
print(f"Calling Gemini: {gemini_llm.model}")

response=gemini_llm.invoke([HumanMessage(content="Explain RAG in 2 sentences.")])

print(response.content)

Calling Gemini: gemini-3.5-flash-lite
23:22:04.782 Chat Completion with 'gemini-3.5-flash-lite' [LLM]
Retrieval-Augmented Generation (RAG) is an AI technique that first searches an external database for relevant information based on a user's prompt. It then feeds that retrieved data into a large language model to generate accurate, context-aware responses.


In [40]:
openai_llm=ChatOpenAI(
    api_key=os.getenv("Open_AI_API"),
    model="gpt-6-luna"

)

In [44]:
del openai_llm

using all 3 in one 

In [43]:

query="What is RAG and Finetuning in 3 sentences."

with logfire.span("LLM_tracing_Simultanously_ all 3 models",query=query, num_models=3):

    #Gemini
    with logfire.span("Gemini_Call",model=gemini_llm.model, provider="Google"):
        t0=time.time()
        try:
            response_gemini=gemini_llm.invoke([HumanMessage(content=query)])
            gemini_ms=round((time.time()-t0)*1000,1)
            logfire.info("Gemini Completed", Latency_ms=gemini_ms,response_length=len(response_gemini.content))
        except Exception as e:
            logfire.warning(f"Gemini Failed. Error: {e}")
    #OpenAI
    with logfire.span("OpenAI Call",model=openai_llm.model,provider="OpenAI"):
        t0=time.time()
        try:
            r_OpenAI=openai_llm.invoke([HumanMessage(content=query)])
            OpenAI_ms=round((time.time()-t0)*1000,1)
            logfire.info("Open AI Completed", latency_openai=OpenAI_ms,response_length=len(r_OpenAI.content))
        except Exception as e:
            logfire.warning(f"Open AI Failed. Error is: {e}")
    #Groq
    with logfire.span("Groq Call", model=llm_groq.model,provider="Groq"):
        
        t0=time.time()

        try:
            r_groq=llm_groq.invoke([HumanMessage(content=query)])
            groq_ms=round((time.time()-t0)*1000,1)
            logfire.info("Groq Completed", latency_groq=groq_ms,response_length=len(r_groq.content))
        except Exception as e:
            logfire.warning("Groq Failed. Error is: {e}")


23:50:03.860 LLM_tracing_Simultanously_ all 3 models
23:50:03.861   Gemini_Call
23:50:03.864     Chat Completion with 'gemini-3.5-flash-lite' [LLM]
23:50:04.907     Gemini Completed
23:50:04.907   OpenAI Call
23:50:04.909     Chat Completion with 'gpt-6-luna' [LLM]
23:50:07.159     Open AI Completed
23:50:07.160   Groq Call
23:50:07.161     Chat Completion with 'meta-llama/llama-prompt-guard-2-86m' [LLM]
23:50:07.395     Groq Completed
